## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [11]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import chromadb
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
#from langchain_community.embeddings import HuggingFaceEmbeddings
import gradio as gr
import os

In [27]:

DB_NAME = "./db_name"
load_dotenv(override=True)

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
#openai_api_url = os.getenv('OPENROUTER_API_URL')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL = "gpt-5-nano"


OpenAI API Key exists and begins sk-proj-


### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [4]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [38]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=1, model_name=MODEL)

### These LangChain objects implement the method `invoke()`

In [31]:
retriever.invoke("Who is Avery?")

[]

In [39]:
llm.invoke("Who is Avery?")

AIMessage(content='Avery is a name, but without context it could mean many things. Do you mean:\n\n- The given name (unisex): origin from Aubrey, from Old Germanic elements meaning “elf ruler” or “noble elf.”\n- A surname that comes from the same root.\n- A specific person or fictional character named Avery (book, show, movie, etc.).\n- A brand or company (Avery Dennison, known for labels and office supplies).\n\nIf you can share a bit more context (are you asking about a person, a character, or the name itself?), I can give a precise answer.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2246, 'prompt_tokens': 10, 'total_tokens': 2256, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 2112, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id

## Time to put this together!

In [33]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [34]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [35]:
answer_question("Who is Averi Lancaster?", [])

'I don’t have information about a person named Averi Lancaster. Could you share more context (industry, organization, a link, or where you heard the name)? I can help look up credible sources or summarize what’s publicly known if you provide a bit more detail. \n\nIf you’re searching on your own, here are quick steps:\n- Check official bios on company or organization websites.\n- Look for LinkedIn profiles or professional directories.\n- Search reputable news outlets or press releases.\n- Cross-check with multiple sources to confirm identity and avoid confusion with similarly named individuals.'

## What could possibly come next? 😂

In [36]:
gr.ChatInterface(answer_question).launch()

/home/andres/Programs/PycharmProjects/ai_bootcamp/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!